# Stylia tutorial

Stylia is a lightweight wrapper around Matplotlib that applies publication-quality styles, color palettes, and figure utilities. This notebook walks through everything it offers.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import stylia

---
## 1. Format and style

Two orthogonal settings control everything:

| Setting | Options | Effect |
|---|---|---|
| `set_format` | `"print"` (default) / `"slide"` | Figure size, font sizes, line widths |
| `set_style`  | `"article"` (default) / `"ersilia"` | Color palette, foreground color (black vs plum) |

Both update `rcParams` globally — call them before creating figures.

In [ ]:
stylia.set_format("print")   # 7.09" wide — Nature two-column width
stylia.set_style("article")  # black structural elements, NPG palette

---
## 2. Named colors

### ArticleColors

Ten perceptually-distinct colors spanning the full hue wheel, plus white and black.

In [ ]:
nc = stylia.ArticleColors()

# Access by name — returns an (R, G, B) tuple in [0, 1]
print(nc.crimson)
print(nc.turquoise)

# Access by index (palette order)
print(nc[0])   # crimson
print(nc[1:3]) # tangerine, amber

In [ ]:
# Modify a color with alpha or lightening
print(nc.get("cobalt", alpha=0.4))    # semi-transparent RGBA
print(nc.get("turquoise", lighten=0.4))  # lightened RGB

In [ ]:
# Visualise the palette
fig, ax = plt.subplots(figsize=(6, 0.5))
ax.axis("off")
for i, color in enumerate(nc):
    ax.add_patch(plt.Rectangle((i, 0), 1, 1, color=color))
ax.set_xlim(0, len(nc))
plt.tight_layout()
plt.show()

### ErsiliaColors

Official Ersilia brand palette — used automatically when `set_style("ersilia")` is active.

In [ ]:
ec = stylia.ErsiliaColors()

print(ec.plum)
print(ec.mint)

fig, ax = plt.subplots(figsize=(5, 0.5))
ax.axis("off")
for i, color in enumerate(ec):
    ax.add_patch(plt.Rectangle((i, 0), 1, 1, color=color))
ax.set_xlim(0, len(ec))
plt.tight_layout()
plt.show()

### NamedColors (style-aware)

`NamedColors` resolves to `ArticleColors` or `ErsiliaColors` based on the active style — useful for style-agnostic code.

In [ ]:
stylia.set_style("article")
nc = stylia.NamedColors()
print(type(nc))  # ArticleColors

stylia.set_style("ersilia")
nc = stylia.NamedColors()
print(type(nc))  # ErsiliaColors

stylia.set_style("article")  # reset

---
## 3. Categorical palettes

`CategoricalPalette` picks maximally-distinguishable colors from a named palette. For n larger than the palette size it interpolates using a colormap.

In [ ]:
from stylia import CategoricalPalette

# Default palettes: "npg" (article style) and "ersilia"
pal_npg     = CategoricalPalette("npg")
pal_ersilia = CategoricalPalette("ersilia")

# Bonus palettes for accessibility / aesthetics
pal_okabe  = CategoricalPalette("okabe")   # Okabe-Ito, colorblind-safe
pal_tol    = CategoricalPalette("tol")     # Paul Tol Bright, colorblind-safe
pal_pastel = CategoricalPalette("pastel")  # soft pastels

In [ ]:
# Get n colors — greedy farthest-point selection for maximum perceptual distance
colors_4 = pal_npg.get(4)
print(colors_4)

# More than palette size: extrapolated by colormap interpolation
colors_15 = pal_npg.get(15)
print(f"{len(colors_15)} colors")

In [ ]:
# Draw one color at a time (stateful counter)
pal = CategoricalPalette("npg")
c1 = pal.next()
c2 = pal.next()
c3 = pal.next()
print(c1, c2, c3)

In [ ]:
# Visualise all palettes
palettes = {
    "npg":     CategoricalPalette("npg"),
    "ersilia": CategoricalPalette("ersilia"),
    "okabe":   CategoricalPalette("okabe"),
    "tol":     CategoricalPalette("tol"),
    "pastel":  CategoricalPalette("pastel"),
}

fig, axes = plt.subplots(len(palettes), 1, figsize=(7, 2.5))
for ax, (name, pal) in zip(axes, palettes.items()):
    colors = pal.get(len(pal._palette))  # all colors
    for i, c in enumerate(colors):
        ax.add_patch(plt.Rectangle((i, 0), 1, 1, color=c))
    ax.set_xlim(0, len(colors))
    ax.set_yticks([])
    ax.set_xticks([])
    ax.set_ylabel(name, rotation=0, labelpad=40, va="center")
plt.tight_layout()
plt.show()

---
## 4. Continuous colormaps

All colormap classes share the same interface:
- `fit(data)` — learn the data range
- `transform(data, alpha=None, lighten=None)` — map data to RGBA colors
- `sample(n)` — get n evenly-spaced colors

### FadingColormap

Near-white → single saturated hue. Good for density or non-negative data.

In [ ]:
from stylia import FadingColormap

data = np.linspace(0, 1, 100)

# Available presets: crimson, cobalt, turquoise, orchid, lime, plum
for preset in ["crimson", "cobalt", "turquoise", "orchid", "lime", "plum"]:
    cmap = FadingColormap(preset)
    cmap.fit(data)
    colors = cmap.sample(8)
    print(f"{preset}: {len(colors)} swatches")

In [ ]:
# Use in a scatter plot
np.random.seed(42)
x = np.random.randn(200)
y = np.random.randn(200)
z = x ** 2 + y ** 2   # distance from origin

cmap = FadingColormap("crimson")
cmap.fit(z)
colors = cmap.transform(z)

fig, ax = plt.subplots(figsize=(3.5, 3))
ax.scatter(x, y, c=colors, s=stylia.MARKERSIZE_SMALL)
ax.set_xlabel("X")
ax.set_ylabel("Y")
ax.set_title("FadingColormap — crimson")
plt.tight_layout()
plt.show()

In [ ]:
# With alpha or lightening
colors_alpha = cmap.transform(z, alpha=0.5)
colors_light = cmap.transform(z, lighten=0.3)

fig, axes = plt.subplots(1, 2, figsize=(7, 2.8))
axes[0].scatter(x, y, c=colors_alpha, s=stylia.MARKERSIZE_SMALL)
axes[0].set_title("alpha=0.5")
axes[1].scatter(x, y, c=colors_light, s=stylia.MARKERSIZE_SMALL)
axes[1].set_title("lighten=0.3")
for ax in axes:
    ax.set_xlabel("X")
    ax.set_ylabel("Y")
plt.tight_layout()
plt.show()

### DivergingColormap

Two hues through a light center. Good for data diverging around a meaningful midpoint (correlations, z-scores, fold changes).

In [ ]:
from stylia import DivergingColormap

# NPG presets: crimson_cobalt, amber_periwinkle
# Ersilia presets: plum_mint, purple_orange

np.random.seed(7)
x = np.random.randn(200)
y = np.random.randn(200)
z = x * 0.8 + np.random.randn(200) * 0.4   # correlated, diverges around 0

dcm = DivergingColormap("crimson_cobalt")
dcm.fit(z)
colors = dcm.transform(z)

fig, ax = plt.subplots(figsize=(3.5, 3))
ax.scatter(x, y, c=colors, s=stylia.MARKERSIZE_SMALL)
ax.set_xlabel("Feature 1")
ax.set_ylabel("Feature 2")
ax.set_title("DivergingColormap — crimson_cobalt")
plt.tight_layout()
plt.show()

In [ ]:
# Flip direction with ascending=False
dcm_flip = DivergingColormap("crimson_cobalt", ascending=False)
dcm_flip.fit(z)
colors_flip = dcm_flip.transform(z)

fig, axes = plt.subplots(1, 2, figsize=(7, 2.8))
for ax, c, title in zip(axes, [colors, colors_flip], ["ascending", "ascending=False"]):
    ax.scatter(x, y, c=c, s=stylia.MARKERSIZE_SMALL)
    ax.set_xlabel("Feature 1")
    ax.set_ylabel("Feature 2")
    ax.set_title(title)
plt.tight_layout()
plt.show()

### SpectralColormap

Multi-hue warm → cool. Good for ordered data where the full range matters (time, rank, a continuous third variable).

In [ ]:
from stylia import SpectralColormap

# NPG preset: npg   (crimson → amber → turquoise → periwinkle → fuchsia)
# Ersilia preset: ersilia  (orange → yellow → mint → blue → purple)

np.random.seed(0)
n = 400
t = np.linspace(0, 1, n)
wx = np.cumsum(np.random.randn(n)) * 0.15
wy = np.cumsum(np.random.randn(n)) * 0.15

scm = SpectralColormap("npg")
scm.fit(t)
colors = scm.transform(t)

fig, ax = plt.subplots(figsize=(3.5, 3))
lw = stylia.LINEWIDTH
for i in range(n - 1):
    ax.plot(wx[i:i+2], wy[i:i+2], color=colors[i], linewidth=lw * 1.5)
ax.set_xlabel("X")
ax.set_ylabel("Y")
ax.set_title("SpectralColormap — npg (colored by time)")
plt.tight_layout()
plt.show()

### CyclicColormap

Wraps back to its starting color. Good for phase or angle data.

In [ ]:
from stylia import CyclicColormap

theta = np.linspace(0, 2 * np.pi, 300)
r = 1 + 0.3 * np.sin(5 * theta)
x_polar = r * np.cos(theta)
y_polar = r * np.sin(theta)

ccm = CyclicColormap("npg")
ccm.fit(theta)
colors = ccm.transform(theta)

fig, ax = plt.subplots(figsize=(3.5, 3))
for i in range(len(theta) - 1):
    ax.plot(x_polar[i:i+2], y_polar[i:i+2], color=colors[i], linewidth=lw * 1.5)
ax.set_xlabel("X")
ax.set_ylabel("Y")
ax.set_title("CyclicColormap — npg (colored by angle)")
plt.tight_layout()
plt.show()

### Sampling swatches

In [ ]:
from stylia import FadingColormap, DivergingColormap, SpectralColormap, CyclicColormap

cmaps = [
    ("crimson",        FadingColormap("crimson")),
    ("plum",           FadingColormap("plum")),
    ("crimson_cobalt",  DivergingColormap("crimson_cobalt")),
    ("plum_mint",       DivergingColormap("plum_mint")),
    ("npg (spectral)",  SpectralColormap("npg")),
    ("ersilia (spectral)", SpectralColormap("ersilia")),
    ("npg (cyclic)",   CyclicColormap("npg")),
    ("ersilia (cyclic)", CyclicColormap("ersilia")),
]

ref = np.linspace(0, 1, 200)

fig, axes = plt.subplots(len(cmaps), 1, figsize=(7, 3))
for ax, (label, cmap) in zip(axes, cmaps):
    cmap.fit(ref)
    colors = cmap.transform(ref)
    for i, c in enumerate(colors):
        ax.add_patch(plt.Rectangle((i, 0), 1, 1, color=c))
    ax.set_xlim(0, len(colors))
    ax.set_yticks([])
    ax.set_xticks([])
    ax.set_ylabel(label, rotation=0, labelpad=70, va="center", fontsize=5)
plt.tight_layout()
plt.show()

---
## 5. Figures

`create_figure` returns a `(fig, axs)` pair. `width` and `height` are fractions of `SIZE` (the format-dependent base width). `axs.next()` steps through panels in order.

In [ ]:
stylia.set_format("print")
stylia.set_style("article")

pal = CategoricalPalette("npg")
nc = stylia.ArticleColors()

fig, axs = stylia.create_figure(1, 2, width=1.0, height=0.4)

# Panel A — line plot
ax = axs.next()
x = np.linspace(0, 4 * np.pi, 120)
for i, col in enumerate(pal.get(3)):
    ax.plot(x, np.sin(x + i) * np.exp(-0.08 * x), color=col,
            linewidth=stylia.LINEWIDTH, label=f"Series {i+1}")
ax.set_xlabel("Time / s")
ax.set_ylabel("Amplitude")
ax.legend(loc="lower right")
stylia.label(ax, title="Damped oscillations", abc="A")

# Panel B — bar plot
ax = axs.next()
labels = ["Alpha", "Beta", "Gamma", "Delta"]
values = np.array([3.8, 2.2, 4.9, 3.1])
err    = np.array([0.3, 0.4, 0.2, 0.5])
ax.bar(labels, values, yerr=err, color=pal.get(4),
       error_kw=dict(linewidth=stylia.LINEWIDTH, capsize=2, capthick=stylia.LINEWIDTH))
ax.set_xlabel("Group")
ax.set_ylabel("Score")
stylia.label(ax, title="Group comparison", abc="B")

stylia.save_figure("/tmp/tutorial_figure.pdf")
plt.show()

### Access panels by index or [row, col]

In [ ]:
fig, axs = stylia.create_figure(2, 2, width=0.9, height=0.7)

# By sequential index
ax = axs[0]
ax.plot([1, 2, 3], color=nc.crimson, linewidth=stylia.LINEWIDTH)
stylia.label(ax, abc="A")

ax = axs[1]
ax.plot([3, 1, 2], color=nc.cobalt, linewidth=stylia.LINEWIDTH)
stylia.label(ax, abc="B")

# By [row, col] — top-right
ax = axs[0, 1]
ax.scatter([1, 2, 3], [3, 1, 2], color=nc.turquoise, s=stylia.MARKERSIZE)
stylia.label(ax, abc="C")

plt.tight_layout()
plt.show()

---
## 6. Ersilia style

Switching to `"ersilia"` style replaces all black structural elements (spines, ticks, labels) with Ersilia plum, and swaps the color cycle to the Ersilia brand palette.

In [ ]:
stylia.set_style("ersilia")

pal_e = CategoricalPalette("ersilia")
nc_e  = stylia.ErsiliaColors()

fig, axs = stylia.create_figure(1, 2, width=1.0, height=0.4)

ax = axs.next()
x = np.linspace(0, 4 * np.pi, 120)
for i, col in enumerate(pal_e.get(3)):
    ax.plot(x, np.sin(x + i) * np.exp(-0.07 * x), color=col,
            linewidth=stylia.LINEWIDTH, label=f"Series {i+1}")
ax.set_xlabel("Time / s")
ax.set_ylabel("Amplitude")
ax.legend(loc="lower right")
stylia.label(ax, title="Ersilia style", abc="A")

ax = axs.next()
dcm = DivergingColormap("plum_mint")
z = np.random.randn(150)
dcm.fit(z)
ax.scatter(np.random.randn(150), np.random.randn(150),
           c=dcm.transform(z), s=stylia.MARKERSIZE_SMALL)
ax.set_xlabel("Feature 1")
ax.set_ylabel("Feature 2")
stylia.label(ax, title="Diverging colormap", abc="B")

plt.show()
stylia.set_style("article")  # reset

---
## 7. Slide format

`set_format("slide")` scales up the canvas to 13" wide and increases font sizes and line widths proportionally.

In [ ]:
stylia.set_format("slide")
stylia.set_style("article")

pal = CategoricalPalette("npg")

fig, axs = stylia.create_figure(1, 2, width=0.6, height=0.35)

ax = axs.next()
x = np.linspace(0, 4 * np.pi, 120)
for i, col in enumerate(pal.get(3)):
    ax.plot(x, np.sin(x + i) * np.exp(-0.08 * x), color=col,
            linewidth=stylia.SLIDE_LINEWIDTH, label=f"Series {i+1}")
ax.set_xlabel("Time / s")
ax.set_ylabel("Amplitude")
ax.legend(loc="lower right")
stylia.label(ax, title="Slide format", abc="A")

ax = axs.next()
labels = ["Alpha", "Beta", "Gamma"]
values = [4.2, 2.8, 5.1]
ax.bar(labels, values, color=pal.get(3))
ax.set_xlabel("Group")
ax.set_ylabel("Score")
stylia.label(ax, title="Bar chart", abc="B")

plt.show()
stylia.set_format("print")  # reset

---
## 8. Sizes and constants

All size constants have `print` and `slide` variants. Use the bare constants when you know the format, or the runtime getters for format-aware code.

In [ ]:
stylia.set_format("print")

# Base width
print(f"SIZE = {stylia.SIZE}")            # 7.09" (print)
print(f"get_size() = {stylia.get_size()}")  # runtime value

# Marker sizes (for scatter s=)
print(f"MARKERSIZE_SMALL = {stylia.MARKERSIZE_SMALL}")
print(f"MARKERSIZE       = {stylia.MARKERSIZE}")
print(f"MARKERSIZE_BIG   = {stylia.MARKERSIZE_BIG}")

# Runtime getter
print(f"get_markersize('small')  = {stylia.get_markersize('small')}")
print(f"get_markersize('normal') = {stylia.get_markersize('normal')}")
print(f"get_markersize('big')    = {stylia.get_markersize('big')}")

# Line widths
print(f"LINEWIDTH       = {stylia.LINEWIDTH}")
print(f"LINEWIDTH_THICK = {stylia.LINEWIDTH_THICK}")

# Font sizes
print(f"FONTSIZE_SMALL = {stylia.FONTSIZE_SMALL}")
print(f"FONTSIZE       = {stylia.FONTSIZE}")
print(f"FONTSIZE_BIG   = {stylia.FONTSIZE_BIG}")

In [ ]:
# Width/height as fractions of SIZE
# create_figure(nrows, ncols, width=1.0, height=0.5)
#   width=1.0  -> full SIZE wide
#   width=0.5  -> half SIZE wide
#   height=0.4 -> 0.4 * SIZE tall

fig, axs = stylia.create_figure(1, 1, width=0.5, height=0.4)
ax = axs.next()
ax.plot([1, 2, 3, 4], [1, 4, 2, 3], color=stylia.ArticleColors().crimson,
        linewidth=stylia.LINEWIDTH)
ax.set_xlabel("X")
ax.set_ylabel("Y")
ax.set_title("Half-width figure")
plt.tight_layout()
plt.show()

---
## 9. Saving figures

In [ ]:
fig, axs = stylia.create_figure(1, 1, width=0.7, height=0.4)
ax = axs.next()
ax.plot([0, 1, 2, 3], [0, 1, 0.5, 1.5], color=stylia.ArticleColors().cobalt,
        linewidth=stylia.LINEWIDTH)
ax.set_xlabel("X")
ax.set_ylabel("Y")
ax.set_title("Output figure")

# save_figure always uses dpi=600, tight bbox — supports .pdf, .png, .svg
stylia.save_figure("/tmp/my_figure.pdf")   # vector PDF for publications
stylia.save_figure("/tmp/my_figure.png")   # raster PNG
print("Saved to /tmp/")